# Setup & Imports

In [1]:
import os
import re
import json
import fitz
import time
import pandas as pd
import tempfile
from pathlib import Path
from dotenv import load_dotenv
from PyPDF2 import PdfReader
from thefuzz import fuzz
from PIL import Image
from typhoon_ocr import ocr_document
from IPython.display import display, Markdown

load_dotenv()
TYPHOON_API_KEY = os.getenv("TYPHOON_OCR_API_KEY")

if not TYPHOON_API_KEY:
    print("⚠️ ไม่พบ TYPHOON_API_KEY กรุณาตรวจสอบไฟล์ .env")
else:
    print("✅ System Ready: Loaded API Key successfully.")

✅ System Ready: Loaded API Key successfully.


# Configuration & Master Data

In [ ]:
# 1. กำหนด Root Directory 
BASE_DIR = Path("../อำเภอบ้านไร่")

# 2. ระบุตำบลที่ต้องการ Process
TARGET_TAMBONS = ["ตำบลแก่นมะกรูด"] # Add your ตำบล

# 3. Master Data สำหรับ Validation (Single Source of Truth)

# 3.1 สำหรับแบบแบ่งเขต (เขตเลือกตั้งที่ 2)
MASTER_CANDIDATES = [
    "นายอรรถพล โต๋วสัจจา",
    "นายชาดา ไทยเศรษฐ์",
    "นางสาวสุชาดา บัวพันธ์",
    "ร.ต.ศึกษา ฟุ้งเฟื่อง ร.น.",
    "นางสุพรรณษา นันทา"
]

# 3.2 สำหรับแบบบัญชีรายชื่อ (รายชื่อพรรคที่ตรวจสอบแล้ว)
MASTER_PARTIES = [
    "ไทยทรัพย์ทวี", "เพื่อชาติไทย", "ใหม่", "มิติใหม่", "รวมใจไทย", 
    "รวมไทยสร้างชาติ", "พลวัต", "ประชาธิปไตยใหม่", "เพื่อไทย", "ทางเลือกใหม่", 
    "เศรษฐกิจ", "เสรีรวมไทย", "รวมพลังประชาชน", "ท้องที่ไทย", "อนาคตไทย",
    "พลังเพื่อไทย", "ไทยชนะ", "พลังสังคมใหม่", "สังคมประชาธิปไตยไทย", "ฟิวชัน", 
    "ไทรวมพลัง", "ก้าวอิสระ", "ปวงชนไทย", "วิชชั่นใหม่", "เพื่อชีวิตใหม่", 
    "คลองไทย", "ประชาธิปัตย์", "ไทยก้าวหน้า", "ไทยภักดี", "แรงงานสร้างชาติ", 
    "ประชากรไทย", "ครูไทยเพื่อประชาชน", "ประชาชาติ", "สรางอนาคตไทย", "รักชาติ", 
    "ไทยพร้อม", "ภูมิใจไทย", "พลังธรรมใหม่", "กรีน", "ไทยธรรม", 
    "แผ่นดินธรรม", "กล้าธรรม", "พลังประชารัฐ", "โอกาสใหม่", "เป็นธรรม", 
    "ประชาชน", "ประชาไทย", "ไทยสร้างไทย", "ไทยก้าวใหม่", "ประชาอาสาชาติ",  
    "พร้อม", "เครือข่ายชาวนาแห่งประเทศไทย", "ไทยพิทักษ์ธรรม", "ความหวังใหม่", "ไทยรวมไทย", 
    "เพื่อบ้านเมือง", "พลังไทยรักชาติ" 
]

# OCR Parser & Validation Engine

In [3]:
import re
from thefuzz import fuzz

class ElectionOCRParser:
    def clean_score_to_int(self, score_str):
        """แปลงคะแนนให้เป็น Int เท่านั้น ถ้าเป็น '-' หรืออักขระอื่น ให้เป็น 0"""
        if not score_str: return 0
        score_str = score_str.strip()
        if score_str in ['-', '—', '.', '']: return 0
        
        # ดึงเฉพาะตัวเลข (Arabic และ Thai)
        digits = re.sub(r'[^\d๑-๙]', '', score_str)
        if digits:
            # แปลงเลขไทยเป็นอารบิกก่อนแปลงเป็น int
            thai_digits = "๐๑๒๓๔๕๖๗๘๙"
            for i, d in enumerate(thai_digits):
                digits = digits.replace(d, str(i))
            return int(digits)
        return 0

    def parse_markdown(self, markdown_text):
        data = {
            "eligible_voters": self.extract_number(r'ผู้มีสิทธิเลือกตั้ง.*?จำนวน\s*([\d,๑-๙]+)', markdown_text),
            "voters_showed_up": self.extract_number(r'มาแสดงตน.*?จำนวน\s*([\d,๑-๙]+)', markdown_text),
            "ballots_allocated": self.extract_number(r'ได้รับจัดสรร.*?จำนวน\s*([\d,๑-๙]+)', markdown_text),
            "ballots_used": self.extract_number(r'บัตรเลือกตั้งที่ใช้.*?จำนวน\s*([\d,๑-๙]+)', markdown_text),
            "valid_ballots": self.extract_number(r'บัตรดี.*?จำนวน\s*([\d,๑-๙]+)', markdown_text),
            "invalid_ballots": self.extract_number(r'บัตรเสีย.*?จำนวน\s*([\d,๑-๙]+)', markdown_text),
            "no_vote_ballots": self.extract_number(r'ไม่เลือก.*?จำนวน\s*([\d,๑-๙]+)', markdown_text),
            "ballots_remaining": self.extract_number(r'บัตรเลือกตั้งที่เหลือ.*?จำนวน\s*([\d,๑-๙]+)', markdown_text),
            "scores": {}
        }
        
        # ค้นหาตาราง (รองรับทั้ง MD และ HTML)
        rows = re.findall(r'\|\s*[\d๑-๙]+\s*\|\s*([^|]+?)\s*\|\s*([^|]+?)\s*\|', markdown_text)
        if not rows:
            rows = re.findall(r'<tr>.*?<td>.*?</td><td>(.*?)</td><td>(.*?)</td>.*?</tr>', markdown_text, re.DOTALL)

        for name, score in rows:
            clean_name = re.sub(r'<[^>]+>', '', name).strip()
            if "ชื่อ" in clean_name or "พรรค" in clean_name: continue
            data['scores'][clean_name] = self.clean_score_to_int(score)
            
        return data

    def validate_data(self, data, file_type, master_list=None, threshold=80):
        flags = {"flag_math_total_used": False, "flag_math_valid_score": False, 
                 "flag_name_mismatch": False, "flag_details": []}
        
        # --- 1. Similarity Mapping ---
        normalized_scores = {}
        if master_list:
            for name, score in data['scores'].items():
                best_match, score_ratio = None, 0
                for m_name in master_list:
                    ratio = fuzz.ratio(name, m_name)
                    if ratio > score_ratio:
                        best_match, score_ratio = m_name, ratio
                
                if score_ratio >= threshold:
                    normalized_scores[best_match] = normalized_scores.get(best_match, 0) + score
                else:
                    normalized_scores[name] = score
                    flags["flag_name_mismatch"] = True
            data['scores'] = normalized_scores

        # --- 2. Math Validation ---
        sum_ballots = data['valid_ballots'] + data['invalid_ballots'] + data['no_vote_ballots']
        if data['ballots_used'] != sum_ballots:
            flags["flag_math_total_used"] = True
            flags["flag_details"].append(f"บัตรใช้({data['ballots_used']}) != รวมย่อย({sum_ballots})")
            
        sum_scores = sum(data['scores'].values())
        if data['valid_ballots'] != sum_scores:
            flags["flag_math_valid_score"] = True
            flags["flag_details"].append(f"บัตรดี({data['valid_ballots']}) != รวมคะแนน({sum_scores})")
            
        flags["needs_manual_check"] = any([flags["flag_math_total_used"], flags["flag_math_valid_score"], flags["flag_name_mismatch"]])
        flags["flag_details"] = " | ".join(flags["flag_details"]) if flags["flag_details"] else "OK"
        return flags

    def extract_number(self, pattern, text):
        match = re.search(pattern, text)
        return self.clean_score_to_int(match.group(1)) if match else 0

# Optimaize .pdf Reader

In [4]:
def process_with_optimization(file_path, file_type):
    doc = fitz.open(file_path)
    full_text = ""
    
    # 🎯 เงื่อนไขแยกประเภท: แบ่งเขตเอาหน้าเดียว (หน้า 0) / บัญชีรายชื่อเอาทุกหน้า
    pages_to_process = [0] if file_type == "แบ่งเขต" else range(len(doc))
    
    for page_idx in pages_to_process:
        page = doc.load_page(page_idx)
        pix = page.get_pixmap(dpi=150)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        
        # 🎯 ถ้าเป็น "แบ่งเขต" ให้หั่นครึ่งบน-ล่าง เพื่อกัน 504 Timeout
        if file_type == "แบ่งเขต":
            print(f"      ✂️ Splitting page {page_idx+1} into chunks to prevent Timeout...")
            w, h = img.size
            chunks = [img.crop((0, 0, w, h//2)), img.crop((0, h//2, w, h))] 
            
            for i, chunk in enumerate(chunks):
                with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
                    chunk.convert('L').save(tmp.name, "JPEG", quality=75) # แปลงขาวดำ+บีบอัด
                    
                    # ระบบ Retry 3 ครั้ง
                    for attempt in range(3):
                        try:
                            full_text += "\n" + ocr_document(pdf_or_image_path=tmp.name)
                            break
                        except Exception as e:
                            print(f"         ⚠️ API Timeout/Error (ครั้งที่ {attempt + 1}) กำลังลองใหม่...")
                            time.sleep(2)
                    os.remove(tmp.name)
        else:
            # 🎯 บัญชีรายชื่อ ส่งทั้งหน้าปกติแต่บีบอัดไฟล์
            with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
                img.convert('L').save(tmp.name, "JPEG", quality=75)
                
                # ระบบ Retry 3 ครั้ง
                for attempt in range(3):
                    try:
                        full_text += "\n" + ocr_document(pdf_or_image_path=tmp.name)
                        break
                    except Exception as e:
                        print(f"         ⚠️ API Timeout/Error (ครั้งที่ {attempt + 1}) กำลังลองใหม่...")
                        time.sleep(2)
                os.remove(tmp.name)
                
    doc.close()
    return full_text

# Data Exporter

## Save Indiv File

In [5]:
def export_individual_result(data, tambon, unit, file_name, format_type='csv'):
    # สร้างโครงสร้าง Folder: output_data/ตำบล.../หน่วย...
    target_dir = Path("output_data") / tambon / unit
    target_dir.mkdir(parents=True, exist_ok=True)
    
    # เปลี่ยนนามสกุลไฟล์จาก .pdf เป็น .csv หรือ .json
    output_filename = Path(file_name).stem + ('.csv' if format_type == 'csv' else '.json')
    save_path = target_dir / output_filename
    
    # บันทึกไฟล์
    if format_type == 'csv':
        # ใช้ [data] เพื่อบังคับให้ Dictionary กลายเป็น 1 แถว (Row) ใน Excel
        df = pd.json_normalize([data]) 
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
    else:
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
            
    return save_path

## Save Summary Report

In [6]:
def export_summary_report(summary_list, format_type='csv'):
    output_dir = Path("output_data")
    output_dir.mkdir(exist_ok=True) # ป้องกันกรณีโฟลเดอร์ยังไม่ถูกสร้าง
    
    df_summary = pd.DataFrame(summary_list)
    save_path = output_dir / f"master_summary_log.{format_type}"
    
    if format_type == 'csv':
        df_summary.to_csv(save_path, index=False, encoding='utf-8-sig')
    else:
        df_summary.to_json(save_path, orient='records', force_ascii=False, indent=4)
        
    print(f"📊 บันทึก Master Summary เรียบร้อยที่: {save_path}")
    
    # โชว์ตัวอย่างตารางใน Notebook ให้ดูหลังรันเสร็จ
    display(Markdown("### 🔍 Preview Summary Data (สำหรับใช้ Query หาจุดผิด)"))
    display(df_summary.head())

# Main Execution Pipeline

In [7]:
parser = ElectionOCRParser()
summary_log = []
EXPORT_FORMAT = 'csv'

In [8]:
print("🚀 Starting Main Pipeline with Optimization & Chunking...")

for tambon_name in TARGET_TAMBONS:
    tambon_dir = BASE_DIR / tambon_name
    if not tambon_dir.exists(): continue
    
    print(f"\n📂 Processing Tambon: {tambon_name}")
    
    for unit_dir in [d for d in tambon_dir.iterdir() if d.is_dir()]:
        unit_name = unit_dir.name
        
        for pdf_file in unit_dir.glob("*.pdf"):
            pdf_path_str = str(pdf_file)
            file_type = "บัญชีรายชื่อ" if "บช" in pdf_file.name else "แบ่งเขต"
            
            print(f"   📄 Reading: {pdf_file.name} ...")
            
            # --- 1. อ่านข้อมูลด้วยฟังก์ชันใหม่ ---
            try:
                raw_md = process_with_optimization(pdf_path_str, file_type)
            except Exception as e:
                print(f"      ❌ PDF Processing Error: {e}")
                continue
            
            # --- 2. Parse & Validate ---
            current_master = MASTER_PARTIES if file_type == "บัญชีรายชื่อ" else MASTER_CANDIDATES
            parsed_data = parser.parse_markdown(raw_md)
            flags_data = parser.validate_data(parsed_data, file_type, master_list=current_master, threshold=80)
            
            # --- 3. รวมข้อมูลเพื่อเซฟรายไฟล์ ---
            full_record = {
                "metadata": {"tambon": tambon_name, "unit": unit_name, "file": pdf_file.name},
                **parsed_data,
                **flags_data
            }
            
            # --- 4. สั่งเซฟแยก Folder ทันที ---
            saved_at = export_individual_result(
                data=full_record, 
                tambon=tambon_name, 
                unit=unit_name, 
                file_name=pdf_file.name, 
                format_type=EXPORT_FORMAT
            )
            print(f"      ✅ Saved individual result to: {saved_at.name}")
            
            # --- 5. เก็บข้อมูลเข้า Master Summary ---
            summary_log.append({
                "tambon": tambon_name,
                "unit": unit_name,
                "type": file_type,
                "file": pdf_file.name,
                "needs_manual_check": flags_data["needs_manual_check"],
                "flag_math_total_used": flags_data["flag_math_total_used"],
                "flag_math_valid_score": flags_data["flag_math_valid_score"],
                "flag_name_mismatch": flags_data["flag_name_mismatch"],
                "details": flags_data["flag_details"]
            })

🚀 Starting Main Pipeline with Optimization & Chunking...

📂 Processing Tambon: ตำบลแก่นมะกรูด
   📄 Reading: ส.ส.5ทับ18 (บช.).pdf ...
      ✅ Saved individual result to: ส.ส.5ทับ18 (บช.).csv
   📄 Reading: ส.ส.5ทับ18.pdf ...
      ✂️ Splitting page 1 into chunks to prevent Timeout...
      ✅ Saved individual result to: ส.ส.5ทับ18.csv
   📄 Reading: ส.ส.5ทับ18 (บช.).pdf ...
      ✅ Saved individual result to: ส.ส.5ทับ18 (บช.).csv
   📄 Reading: ส.ส.5ทับ18.pdf ...
      ✂️ Splitting page 1 into chunks to prevent Timeout...
      ✅ Saved individual result to: ส.ส.5ทับ18.csv
   📄 Reading: ส.ส.5ทับ18 (บช.).pdf ...
      ✅ Saved individual result to: ส.ส.5ทับ18 (บช.).csv
   📄 Reading: ส.ส.5ทับ18.pdf ...
      ✂️ Splitting page 1 into chunks to prevent Timeout...
      ✅ Saved individual result to: ส.ส.5ทับ18.csv
   📄 Reading: ส.ส.5ทับ18 (บช.).pdf ...
      ✅ Saved individual result to: ส.ส.5ทับ18 (บช.).csv
   📄 Reading: ส.ส.5ทับ18.pdf ...
      ✂️ Splitting page 1 into chunks to prevent Timeout.

# Call Function

In [9]:
print("\n🎉 All PDF Processed. Saving Master Summary...")
export_summary_report(summary_list=summary_log, format_type=EXPORT_FORMAT)


🎉 All PDF Processed. Saving Master Summary...
📊 บันทึก Master Summary เรียบร้อยที่: output_data/master_summary_log.csv


### 🔍 Preview Summary Data (สำหรับใช้ Query หาจุดผิด)

,tambon,unit,type,file,needs_manual_check,flag_math_total_used,flag_math_valid_score,flag_name_mismatch,details
0,ตำบลแก่นมะกรูด,หน่วยเลือกตั้งที่ 1,บัญชีรายชื่อ,ส.ส.5ทับ18 (บช.).pdf,True,False,True,True,บัตรดี(270) != รวมคะแนน(267)
1,ตำบลแก่นมะกรูด,หน่วยเลือกตั้งที่ 1,แบ่งเขต,ส.ส.5ทับ18.pdf,True,False,True,True,บัตรดี(254) != รวมคะแนน(0)
2,ตำบลแก่นมะกรูด,หน่วยเลือกตั้งที่ 2,บัญชีรายชื่อ,ส.ส.5ทับ18 (บช.).pdf,True,True,True,True,บัตรใช้(285) != รวมย่อย(0) | บัตรดี(0) != รวมค...
3,ตำบลแก่นมะกรูด,หน่วยเลือกตั้งที่ 2,แบ่งเขต,ส.ส.5ทับ18.pdf,True,True,True,True,บัตรใช้(300) != รวมย่อย(250) | บัตรดี(200) != ...
4,ตำบลแก่นมะกรูด,หน่วยเลือกตั้งที่ 4,บัญชีรายชื่อ,ส.ส.5ทับ18 (บช.).pdf,True,False,True,True,บัตรดี(321) != รวมคะแนน(374)
